# Embedding a view in your website

Use this page when you want an interactive 3D view on a website you control: a
documentation site, a rendered notebook, a project page, or material attached to
a paper.

It works the same way everywhere, because what you produce is an ordinary HTML
file plus one shared asset. Sphinx, MkDocs, Quarto, Jupyter Book, Hugo and plain
HTML all embed it identically. The Sphinx section near the end is a convenience,
not a requirement.

## The short version

```python
import molsysviewer as msv

view = msv.new_view("1TCD")
view.export.html("docs/_static/views/1tcd.html", shared_runtime="docs/_static")
```

That writes two things:

- `docs/_static/views/1tcd.html` — your scene, as inline JSON;
- `docs/_static/viewer.js` — the MolSysViewer runtime, copied from your
  installation.

Then embed the view in whatever page you like:

```python
print(msv.tools.embed_iframe(
    "docs/_static/views/1tcd.html",
    path="docs/content/my_page.md",
))
```

```html
<iframe src="../_static/views/1tcd.html" width="100%" height="480px"
        style="border:none;"></iframe>
```

No server, no Python kernel and no build system are needed to view the result.
Open the HTML file and it renders.

## One question: shared or self-contained?

That is the whole decision, and `shared_runtime` is the whole API for it.

```python
view.export.html(path)                                  # self-contained
view.export.html(path, shared_runtime="docs/_static")   # shared with your other views
```

**Without `shared_runtime`** you get a single file that carries everything —
several megabytes, and it goes anywhere on its own. Right for something you send
to a colleague or attach to a paper.

**With a directory** you get a small file that loads the runtime from a copy
placed there. Right for a website: the runtime is downloaded once and cached
across every page, however many views you publish.

The copy comes from the MolSysViewer you have installed, which is what makes it
trustworthy:

- it is **version-exact with your scene** — the same code that produced it;
- your page **works offline**, and on networks that block external hosts;
- it keeps working regardless of what is published anywhere later;
- and when you upgrade MolSysViewer and regenerate, you get the new runtime.

### If the views and the asset are produced by different steps

Place the asset on its own:

```python
import molsysviewer as msv

msv.tools.export_runtime_asset("docs/_static")
```

Useful in a `Makefile` or CI step that prepares assets before, or independently
of, generating scenes.

The copy is replaced when you upgrade MolSysViewer and left alone when it is
already current, so a build that watches timestamps does not churn. A
`viewer.js` in that directory that MolSysViewer did not write is **never**
overwritten — the export stops and tells you, rather than destroying a bundle of
your own that happens to share the name.

### Should the asset be committed?

Both policies work; it is your repository's call.

- **Commit it** — anyone who clones can open the views directly, with no build.
- **Generate it at build time and `.gitignore` it** — no large binary in your
  history. Your CI already has MolSysViewer installed if it generates the views,
  so placing the asset is one more line.

MolSysViewer's own documentation uses the second: `docs/conf.py` calls
`export_runtime_asset` when the build starts.

### Not hosting anything at all

If you would rather not keep a runtime beside your views, address the published
package:

```python
view.export.html(path, shared_runtime="cdn")
```

This requires your MolSysViewer version to be a published release — exporting
from a development install raises, because the URL would be written now and fail
later, on your readers' screens. Weigh it knowing that the page then depends on
that version remaining available for as long as the page is read, and will not
render without a network connection.

You can also give an explicit list of candidates, tried in order:

```python
view.export.html(path, shared_runtime=["../viewer.js", "https://…/viewer.js"])
```

*(A directory literally named `cdn` must be written as `"./cdn"` or as an
absolute path.)*

## The two shapes side by side

|  | `shared_runtime="<dir>"` | no `shared_runtime` |
|---|---|---|
| Runtime | shared, one copy for all views | inside every file |
| File size | small | several MB each |
| Works offline | yes | yes |
| Best for | a site with more than one view | a single file you send to someone |

## What an embedded view can and cannot do

The exported page is the scene plus the renderer. There is no Python behind it,
which is what lets it work anywhere — and it is also what sets the boundary.

**Works:** everything already in the scene (structures, regions, colours,
representations, overlays, annotations, measurements), full camera control,
reset, fullscreen, spin, swing, trajectory playback, and the pop-out window.

**Does not:** anything that needs the Python side to decide something — sending a
selection back to a notebook, `on_click` / `on_hover` callbacks, loading new
data, or recording new reproducible state. Compose the scene you want before
exporting it.

## A worked example

Let's actually produce a file. Build a view and export it:

In [1]:
import molsysviewer as msv

view = msv.new_view('1TCD')
view.export.html('1tcd_view.html', title='1TCD View', shared_runtime='.')

Place the result in a directory reserved for it in your documentation tree —
we suggest `docs/_static/views/` — and point every view at one shared runtime
directory, as shown above.

By default the exported view includes the on-canvas controls (Reset, Fullscreen,
background toggle, Spin, Swing). For a more discreet embed, pass
`include_controls=False`.

During a Sphinx build everything under `_static/` is copied to the output, so
both the view and the runtime are served automatically.

## Embedding the result

Counting `../` by hand is the one step of embedding that fails silently: the
export succeeds, the build succeeds, and the reader gets an empty frame. So let
the code count. Give it the view and the page that will show it:

```python
import molsysviewer as msv

print(msv.tools.embed_iframe(
    "docs/_static/views/1tcd.html",
    path="docs/content/user/my_page.md",
    height="480px",
))
```

Paste the result into the page. It works unchanged in Sphinx, MkDocs, Quarto,
Jupyter Book or plain HTML.

Inside a notebook, `load_html_in_notebook` embeds a view as an output cell.
Here it shows one generated by this project's own `docs/generate_static_views/`
scripts:

In [2]:
from molsysviewer.thirds.jupyter import load_html_in_notebook

load_html_in_notebook('../../../_static/views/demo_1TCD.html')

The relative path is resolved from the **built** page's location, so count
directories from the output HTML rather than from the notebook source.

Leave a blank cell below the embed so the view has some visual space.

## Notebook tips: showing the view, not the plumbing

If you write the page as a notebook, two cell tags keep the result clean while
the notebook stays fully reproducible.

**Hide the output of `view.show()`.** Select the cell, open the Property
Inspector (the gear icon), and add the cell tag `remove-output`. `myst_nb` keeps
the input in the source and hides the output in the rendered HTML.

**Hide the input of the embedding call.** Select the `load_html_in_notebook`
cell and add the tag `remove-input`, so readers see the view without the
mechanics that produced it.

The result is a clean interactive scene on the page, with the full code still
present underneath.

## Generate views outside the documentation build

Export scenes with a script, not from a notebook executed during the build:

```python
# docs/generate_static_views/1tcd.py
import molsysviewer as msv

view = msv.demo["1TCD"]
view.styles.apply(tag="polymer-and-ligand")
view.export.html("../_static/views/1tcd.html", shared_runtime="../_static")
```

Run them when the scene changes, and commit the result. Keeping generation out of
the build is what makes the build fast and reproducible without a Jupyter
kernel.

## Sphinx and MyST specifics

Everything above applies unchanged. In a `.md` page, embed with a `raw`
directive:

````markdown
```{raw} html
<iframe src="../../../_static/views/1tcd.html" width="100%" height="480"
        style="border:none;"></iframe>
```
````

Then run your usual `make html`. Open the generated pages and check that the
scene renders and that any `show()` output you meant to hide is hidden.

## See also

- {doc}`html_export` — every option of `view.export.html`.
- {doc}`../../developer/documentation/web/build_and_layout` — how this project's
  own documentation is laid out and built.

In [3]:
# A lite export writes two files: the view and the shared runtime.
!rm -f 1tcd_view.html viewer.js